In [ ]:
using CSV
using DataFrames
using DataFramesMeta
using LinearAlgebra
using Statistics

In [ ]:
# functions to merge the markov pairs or "dictionaries" into a single dictionary
function get_merged_pairs(s_df::DataFrame; funhar=false, unique_char=false, n_chunks=10)
    if funhar

        key_seqs = map(x -> convert(Array{Any,1}, split(x, "-")), s_df[!, :FunHar_Sequence])
    else
        key_seqs = map(x -> convert(Array{Any,1}, split(x, "-")), s_df[!, :Key_Sequence])
    end
    if unique_char
        markov_pairs = map(x -> get_markov_pairs(rle(x)[1]), key_seqs)
    else
        markov_pairs = map(x -> get_markov_pairs(x), key_seqs)
    end
    if length(markov_pairs) > 2000
        chunked_mp = chunk(markov_pairs, n_chunks)
        merged_cmp = map(x -> mergewith(+, x...), chunked_mp)
        return mergewith(+, merged_cmp...)
    else
        return mergewith(+, markov_pairs...)
    end
end
function get_merged_pairs(k_s::Array{String,1}; unique_char=false)
    key_seqs = map(x -> convert(Array{Any,1}, split(x, "-")), k_s)
    if unique_char
        markov_pairs = map(x -> get_markov_pairs(rle(x)[1]), key_seqs)
    else
        markov_pairs = map(x -> get_markov_pairs(x), key_seqs)
    end
    return merge(+, markov_pairs...)
end
function get_merged_pairs(k_s::String; unique_char=false)
    key_seqs = convert(Array{Any,1}, split(k_s, "-"))
    if unique_char
        markov_pairs = get_markov_pairs(rle(key_seqs)[1])
    else
        markov_pairs = get_markov_pairs(key_seqs)
    end
    return markov_pairs
end
chunk(arr, n) = [arr[i:min(i + n - 1, end)] for i in 1:n:length(arr)]
β = 1 / 100000

In [ ]:
function get_key_uncertainty(dis::Vector{Float64}; λ=12)
    # Process the input vector if it's length 3
    if length(dis) == 3
        dis = msr2.get_distance_to_keys(dis)[:, 2][1:12]
    end

    # computing the probability distribution from the distances (softmax or boltzmann distribution)
    prob_d = map(x -> exp(-x * λ), dis)
    prob_d = prob_d ./ sum(prob_d)
    if sum(prob_d) == 0 || isnan(sum(prob_d))
        prob_d = [1 / length(dis) for i = 1:length(dis)]
    end
    return mapreduce(x -> -x * log2(x), +, prob_d)
end

In [ ]:
#building models
"""
    get_markov_pairs(c)

    Returns the counts for the elements and pairs of consecutive elements of
    a given sequence c.

    The output are two dictionaries, each of them having symbols as keys
    and counts as values.
"""
function get_markov_pairs(c)
    #Tm = Dict{Any,Int64}()
    T = Dict{Any,Int64}()
    xy = join(["***" c[1]], "+") #first element.
    yz = join([c[end] "***"], "+") #last element.
    T[xy] = get(T, xy, 0) + 1
    T[yz] = get(T, yz, 0) + 1
    #Tm[c[1]] = get(Tm,c[1],0)+1
    for i = 1:length(c)-1
        #Tm[c[i+1]] = get(Tm,c[i+1],0)+1
        xy = join([c[i] c[i+1]], "+")
        T[xy] = get(T, xy, 0) + 1
    end
    return T# Tm
end

"""
    KLD_rate(piece,prev_pool,beta)

    Computes the Kullback-Leibler Divergence rate between two stochastic matrices:
    KL(P||Q) = Σ μ p(x)log(p(x)/q(x)), where μ is the asymptotic distribution of marginals.
    Beta is a parameter for additive smoothing of the distribution Q.
"""
function KLD_rate(piece, prev_pool, beta)
    μ_i, P_ij = get_tprobs(piece) #asymptotic distribution of marginals, transition probabilities
    Q_prev = get_tprobs_prev(piece, prev_pool, beta)
    P = collect(values(P_ij))[2:end] #distribution of the transitions in the piece
    Q = collect(values(Q_prev))[2:end] #probabilites of the transitions for the piece in pool
    #KL divergence rate, K(p|q) = Σ μ p(x)log(p(x)/q(x)P
    return mapreduce((x, y, z) -> z * x * log10(x / y), +, P, Q, μ_i)
end

"""
    get_tprobs_prev(f_eij, prev_eij, beta)

    Returns a dictionary of probabilites (normalized frequency) from a dictionary
    f_eij and the dictionary of counts prev_eij (previous pool), using a Laplacian
    smoothing within the transition space n_states or PT_exj.
"""
function get_tprobs_prev(f_eij, prev_eij, beta)
    f_ei = Dict{Any,Int64}()
    for eij in keys(prev_eij)
        ei = split(eij, "+")[1]
        f_ei[ei] = get(f_ei, ei, 0) + prev_eij[eij]
    end
    p_ij = Dict{Any,Float64}()

    for eij in keys(f_eij)
        e1 = split(eij, "+")[1]
        if haskey(prev_eij, eij)
            prob_ij = (prev_eij[eij] + beta) / (f_ei[e1] + beta * 24) #case when the transition exists
        elseif haskey(f_ei, e1) #transition doesnt exists but first token does
            prob_ij = beta / (f_ei[e1] + beta * 24)
        else
            prob_ij = beta / (beta * 24) #completely new pair of elements
        end
        p_ij[eij] = get(p_ij, eij, prob_ij)
    end

    return sort(p_ij)
end
"""
    get_tprobs(f_eij)

    Returns probabilities (normalized frequencies) from a dictionary of transition
    counts, the transitions are normalized by the transitions of the first state,
    constructing an stochastic matrix without a matrix.
"""
function get_tprobs(f_eij)
    f_ei = Dict{Any,Int64}()
    p_ei = Dict{Any,Float64}()
    n_ei = []
    for eij in keys(f_eij)
        ei = split(eij, "+")[1]
        push!(n_ei, ei)
        f_ei[ei] = get(f_ei, ei, 0) + f_eij[eij]
        p_ei[ei] = get(f_ei, ei, 0) + f_eij[eij]
    end
    p_e = Dict{Any,Float64}()
    n_e = sum(values(f_ei))
    for e in keys(f_ei)
        prob_e = f_ei[e] / n_e
        p_e[e] = get(p_e, e, prob_e)
    end
    m_p = Float64[]
    p_ij = Dict{Any,Float64}()
    for eij in keys(f_eij)
        e1 = split(eij, "+")[1]
        prob_ij = f_eij[eij] / f_ei[e1]
        p_ij[eij] = get(p_ij, eij, prob_ij)
    end
    p_ij = sort(p_ij)
    for eij in keys(p_ij)
        e1 = split(eij, "+")[1]
        push!(m_p, p_e[e1])
    end
    return m_p, sort(p_ij)
end


# Importing the [Spiral Representation package](https://github.com/spiralizing/MusicSpiralRepresentation.jl)

In [ ]:
import MusicSpiralRepresentation as msr2

In [ ]:
# function to get the sequence of numerals, using as reference the most common key in the sequence.
function get_num_seq(kseq::Vector{String})
    g_key = msr2.get_rank_freq(kseq)[1,1] #most common key in the sequence
    num_seq = msr2.funhar_seq(kseq, g_key) #transform the sequence of keys into a sequence of numerals, using the most common key as reference
    return num_seq
end

## Loading utility functions

In [ ]:
# defining the directory (current) for the data and utils
dir = pwd()

In [ ]:
const srt = include(joinpath(dir, "msr2_utils.jl"))

## Loading the pieces information (csv with the extracted information from the MIDI files)

In [ ]:
#pieces info; piece id
pieces_df = CSV.read(joinpath(dir, "Data/Pieces_Metrics.csv"), DataFrame) # your Data folder
#removing duplicate
deleteat!(pieces_df, findfirst(pieces_df[!, :Piece_ID] .== "kdf_5095"));
piece_id = pieces_df[!, :Piece_ID]
uyears = unique(pieces_df[!, :Year]); #unique years in the dataset

In [ ]:
csv_folder = joinpath(dir, "Data/CSVPieces") #your Data folder

In [ ]:
csv_files =  map(x-> split(x, ".")[1], readdir(csv_folder));

# Computing the sequences of center of effects, keys and numerals for each piece

In [ ]:
# initialize the variables
seq_coes = [];
numeral_seqs = Vector{Vector{String}}()
key_seqs = Vector{Vector{String}}()

# cycle over the pieces 
for np in 1:length(piece_id)
    file = findfirst(x-> x == piece_id[np], csv_files)
    df_piece = CSV.read(joinpath(csv_folder, string(csv_files[file], ".csv")), DataFrame, header=true);
    try 
        meas_piece = groupby(df_piece, :Measure)
        piece_keys = String[]
        piece_coes = []
        for i in 1:length(meas_piece)
            piece_coe = msr2.get_center_effect(Matrix(meas_piece[i]))
            push!(piece_coes, piece_coe)
            key_meas = msr2.get_distance_to_keys(piece_coe)[1, 1]
            push!(piece_keys, key_meas)
        end
        num_meas = get_num_seq(piece_keys)
        push!(seq_coes, piece_coes)
        push!(numeral_seqs, num_meas)
        push!(key_seqs  , piece_keys)
    catch
        meas_piece = groupby(df_piece, :Measure)[1:end-1]
        piece_keys = String[]
        piece_coes = []
        for i in 1:length(meas_piece)
            piece_coe = msr2.get_center_effect(Matrix(meas_piece[i]))
            push!(piece_coes, piece_coe)
            key_meas = msr2.get_distance_to_keys(piece_coe)[1, 1]
            push!(piece_keys, key_meas)
        end
        num_meas = get_num_seq(piece_keys)
        push!(seq_coes, piece_coes)
        push!(numeral_seqs, num_meas)
        push!(key_seqs  , piece_keys)
    end

end

# Building markov models

In [ ]:
# constructing the markov pairs for the sequences of numerals and keys
markov_numerals = map(x -> get_markov_pairs(x), numeral_seqs);
markov_keys = map(x -> get_markov_pairs(x), key_seqs);

In [ ]:
pool_key_markov = Array{Dict{Any,Int64}}(undef, length(uyears))
pool_key_markov[1] = copy(markov_keys[1])
for y in 2:length(uyears)
    ixs = findall(x -> x == uyears[y], pieces_df[!, :Year])
    pool_key_markov[y] = mergewith(+, pool_key_markov[y-1], markov_keys[ixs]...)
end

In [ ]:
pool_numeral_markov = Array{Dict{Any,Int64}}(undef, length(uyears))
pool_numeral_markov[1] = copy(markov_numerals[1])
for y in 2:length(uyears)
    ixs = findall(x -> x == uyears[y], pieces_df[!, :Year])
    pool_numeral_markov[y] = mergewith(+, pool_numeral_markov[y-1], markov_numerals[ixs]...)
end

# Computing Novelty measure

### For keys:

In [ ]:
key_novelties = Array{Float64}(undef, length(markov_keys));
for np in 2:length(markov_keys)
    #finds the index of the year previous to the piece year
    ix_yr = maximum(findall(x -> x < pieces_df[np, :Year], uyears))
    # selects the transition pool that corresponds to the years previous to the piece year
    prev_pool = copy(pool_key_markov[ix_yr])
    #calculates novelty
    key_novelties[np] = KLD_rate(markov_keys[np], prev_pool, β)
end

### For transposed keys (numerals):

In [ ]:
numeral_novelties = Array{Float64}(undef, length(markov_numerals));
for np in 2:length(markov_numerals)
    #finds the index of the year previous to the piece year
    ix_yr = maximum(findall(x -> x < pieces_df[np, :Year], uyears))
    # selects the transition pool that corresponds to the years previous to the piece year
    prev_pool = copy(pool_numeral_markov[ix_yr])
    #calculates novelty
    numeral_novelties[np] = KLD_rate(markov_numerals[np], prev_pool, β)
end

In [ ]:
out_csv = DataFrame(Piece_ID=pieces_df[!, :Piece_ID],
    Year=pieces_df[!, :Year],
    Composer=pieces_df[!, :Composer],
    Novelty_original=key_novelties,
    Novelty_numeral=numeral_novelties)

## Saving the .csv file

In [ ]:
CSV.write("Pieces_data.csv", out_csv)